# Do some line-ups win more than their agents deserve?

The project's other question. Some agents might work better *together* than their own
records suggest. If so, a line-up should win more than its parts predict.

**The trap:** a line-up winning 70% could mean the agents combine well, or that one very good
team likes it. The raw numbers look identical, and the second is far more likely. Most of
this notebook is about telling them apart.

## First: what could this data possibly show?

Settled before looking at any result, so nothing gets over-read afterwards.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from dataset import build_map_table, season_folds, split_season
from features import build_features, feature_columns
from form import player_form

maps = build_map_table()

print("A line-up used N times, which truly wins half its maps.")
print("How far from 50% must it land before we can call that real rather than luck?\n")
for n in [10, 20, 40, 86]:
    margin = 1.96 * np.sqrt(0.25 / n) * 100
    print(f"   used {n:3} times  ->  needs to beat {50 + margin:4.1f}%  (margin +/-{margin:.0f} points)")

print("\nTurned around: how many maps would we need to spot a genuine edge?\n")
for edge in [0.02, 0.05, 0.10, 0.15]:
    needed = ((1.96 + 0.84) ** 2 * 0.25) / edge ** 2
    print(f"   a real {edge * 100:4.1f} point edge needs about {needed:5.0f} maps of that exact line-up")

A line-up used N times, which truly wins half its maps.
How far from 50% must it land before we can call that real rather than luck?

   used  10 times  ->  needs to beat 81.0%  (margin +/-31 points)
   used  20 times  ->  needs to beat 71.9%  (margin +/-22 points)
   used  40 times  ->  needs to beat 65.5%  (margin +/-15 points)
   used  86 times  ->  needs to beat 60.6%  (margin +/-11 points)

Turned around: how many maps would we need to spot a genuine edge?

   a real  2.0 point edge needs about  4900 maps of that exact line-up
   a real  5.0 point edge needs about   784 maps of that exact line-up
   a real 10.0 point edge needs about   196 maps of that exact line-up
   a real 15.0 point edge needs about    87 maps of that exact line-up


**The key number:** a realistic synergy is worth maybe 2 to 5 points, and detecting 5 points
needs about **784 maps of one exact line-up**. The most-used line-up in 2025 appears **86
times**.

So, decided in advance: **this data can only detect enormous effects, of 15 points or more.**
Finding nothing wouldn't mean synergy doesn't exist, and anything big enough to see is more
likely a good team than a good line-up.

## 1. Every line-up, and each agent's own record

In [2]:
# Every map contributes two rows, one per side, each with the line-up that side
# played and whether it won.
sides = pd.concat([
    maps[["comp_a", "map", "team_a", "team_a_won", "match_id"]]
      .rename(columns={"comp_a": "comp", "team_a": "team", "team_a_won": "won"}),
    maps[["comp_b", "map", "team_b", "team_a_won", "match_id"]]
      .assign(won=lambda d: 1 - d["team_a_won"])
      .rename(columns={"comp_b": "comp", "team_b": "team"}),
], ignore_index=True)

print(f"{len(sides):,} line-up appearances, from {len(maps):,} maps")

usage = sides.groupby("comp").size().sort_values(ascending=False)
print(f"{usage.size} different line-ups; the most-used appears {usage.iloc[0]} times")
print(f"{(usage == 1).sum()} were used exactly once ({(usage == 1).mean():.0%})\n")

# each agent's own record, across every map it appeared on
agent_records = (
    pd.DataFrame([{"agent": agent, "won": won}
                  for comp, won in zip(sides["comp"], sides["won"])
                  for agent in comp])
    .groupby("agent")["won"].agg(rate="mean", maps="size")
)

print("agent win rates must average near 50% -- every map has a winner and a loser:")
print(f"   {agent_records['rate'].min():.1%} to {agent_records['rate'].max():.1%}, "
      f"averaging {agent_records['rate'].mean():.1%}")

2,544 line-up appearances, from 1,272 maps
459 different line-ups; the most-used appears 86 times
162 were used exactly once (35%)

agent win rates must average near 50% -- every map has a winner and a loser:
   38.5% to 55.9%, averaging 49.8%


## 2. Expected against actual

A line-up's expected win rate is the average of its five agents' own win rates; the gap is
the over-performance. Only line-ups used **10+ times** are considered.

In [3]:
MIN_USES = 10
rate_of = agent_records["rate"].to_dict()

lineups = sides.groupby("comp").agg(uses=("won", "size"), actual=("won", "mean"),
                                    teams=("team", "nunique"))
lineups["expected"] = [np.mean([rate_of[a] for a in comp]) for comp in lineups.index]
lineups["over"] = lineups["actual"] - lineups["expected"]

busy = lineups[lineups["uses"] >= MIN_USES].copy()
busy["wins"] = (busy["actual"] * busy["uses"]).round().astype(int)
busy["p"] = [stats.binomtest(int(w), int(n), float(e)).pvalue
             for w, n, e in zip(busy["wins"], busy["uses"], busy["expected"])]

print(f"{len(busy)} line-ups used at least {MIN_USES} times, "
      f"covering {busy['uses'].sum():,} of {len(sides):,} appearances\n")
print(f"   over-performing by more than 5 points : {(busy['over'] > 0.05).sum()}")
print(f"   under-performing by more than 5 points: {(busy['over'] < -0.05).sum()}")
print()
print(f"   look 'significant' at the usual 0.05 threshold: {(busy['p'] < 0.05).sum()}")
print(f"   expected from pure chance across {len(busy)} tests : {0.05 * len(busy):.1f}")
print(f"   survive a correction for testing {len(busy)} things  : "
      f"{(busy['p'] < 0.05 / len(busy)).sum()}")

top = busy.nlargest(5, "over")[["uses", "teams", "expected", "actual", "over", "p"]]
top.index = [", ".join(c) for c in top.index]
print("\nthe five biggest over-performers:")
print(top.to_string(float_format=lambda v: f"{v:.3f}"))

54 line-ups used at least 10 times, covering 1,460 of 2,544 appearances

   over-performing by more than 5 points : 17
   under-performing by more than 5 points: 16

   look 'significant' at the usual 0.05 threshold: 2
   expected from pure chance across 54 tests : 2.7
   survive a correction for testing 54 things  : 0

the five biggest over-performers:
                                       uses  teams  expected  actual  over     p
kayo, omen, sova, viper, waylay          12      2     0.498   0.917 0.419 0.003
astra, fade, raze, viper, yoru           23      4     0.509   0.739 0.230 0.035
fade, omen, raze, vyse, yoru             43     10     0.509   0.651 0.143 0.068
brimstone, fade, raze, viper, yoru       28     13     0.509   0.643 0.133 0.187
breach, brimstone, cypher, neon, tejo    41     20     0.493   0.610 0.117 0.160


**Two line-ups look significant, and chance alone gives 2.7.** Testing 54 things at a
1-in-20 threshold throws up a few by luck, and none survive a correction for that. About as
many under-perform as over-perform: noise around zero.

## 3. The trap, made visible

Two of the strongest-looking line-ups, by who played them:

In [4]:
for comp in [("fade", "omen", "raze", "vyse", "yoru"),
             ("kayo", "omen", "sova", "viper", "waylay")]:
    sub = sides[sides["comp"] == comp]
    by_team = (sub.groupby("team")
                 .agg(uses=("won", "size"), wins=("won", "sum"), rate=("won", "mean"))
                 .sort_values("uses", ascending=False))
    print(f"{', '.join(comp)}")
    print(f"   {len(sub)} appearances, {sub['won'].sum()} wins ({sub['won'].mean():.1%} overall)")
    print(by_team.head(6).to_string(float_format=lambda v: f"{v:.2f}"))
    print()

fade, omen, raze, vyse, yoru
   43 appearances, 28 wins (65.1% overall)
               uses  wins  rate
team                           
Paper Rex        20    15  0.75
EDward Gaming     5     1  0.20
Team Liquid       4     4  1.00
TALON             3     3  1.00
Apeks             2     1  0.50
100 Thieves       2     1  0.50

kayo, omen, sova, viper, waylay
   12 appearances, 11 wins (91.7% overall)
      uses  wins  rate
team                  
NRG      8     8  1.00
MIBR     4     3  0.75



The first wins 65% overall, but **Paper Rex play 20 of its 43 maps and win 75%**, while
EDward Gaming went 1–4 with the same five agents. The second wins 92% because **NRG played it
eight times and won all eight.** Those are facts about Paper Rex and NRG, not the agents.

## 4. A test one strong team can't fake

- **Judge each line-up against what the model expected**, not against 50%. Chemistry the model
  can't see would make sides beat their predictions. Every prediction comes from a model that
  never saw that map.
- **Count each team once.** Measure how far each team beat its predictions with a line-up,
  then ask whether that leans positive across different teams. A line-up needs at least three
  teams using it at least twice each.

In [5]:
learn, _ = split_season(maps)
COLUMNS = feature_columns(maps)
X_agents, y = build_features(learn, COLUMNS)
form = player_form(learn, half_life=5).fillna(0.0)
X = pd.concat([X_agents, form], axis=1)
MODEL = ["rating_diff", "map_picked_by_a"]

# out-of-fold predictions: every map is predicted by a model that never saw it
predicted = pd.Series(np.nan, index=learn.index)
for learn_rows, check_rows in season_folds(learn):
    x_train, x_check = X.loc[learn_rows, MODEL], X.loc[check_rows, MODEL]
    middle, spread = x_train.mean(), x_train.std().replace(0, 1)
    model = LogisticRegression(max_iter=5000).fit((x_train - middle) / spread, y.loc[learn_rows])
    predicted.loc[check_rows] = model.predict_proba((x_check - middle) / spread)[:, 1]

scored = learn[predicted.notna()].copy()
scored["p_a"] = predicted[predicted.notna()]

# how much each side beat what was expected of it
beat = pd.concat([
    pd.DataFrame({"comp": scored["comp_a"], "team": scored["team_a"],
                  "beat_expectation": scored["team_a_won"] - scored["p_a"]}),
    pd.DataFrame({"comp": scored["comp_b"], "team": scored["team_b"],
                  "beat_expectation": (1 - scored["team_a_won"]) - (1 - scored["p_a"])}),
], ignore_index=True)

print(f"{len(beat):,} scored appearances from {len(scored):,} maps\n")

# --- counting every appearance separately, the wrong way ---
naive = beat.groupby("comp")["beat_expectation"].agg(avg="mean", n="size")
naive = naive[naive["n"] >= MIN_USES]
naive["p"] = [stats.ttest_1samp(beat.loc[beat["comp"] == c, "beat_expectation"], 0).pvalue
              for c in naive.index]

print(f"COUNTING EVERY APPEARANCE SEPARATELY  ({len(naive)} line-ups)")
print(f"   significant at 0.05 : {(naive['p'] < 0.05).sum()}   "
      f"(chance would give {0.05 * len(naive):.1f})")
print(f"   survives correction : {(naive['p'] < 0.05 / len(naive)).sum()}")

# --- counting each team once ---
per_team = (beat.groupby(["comp", "team"])["beat_expectation"]
                .agg(avg="mean", n="size").reset_index())
per_team = per_team[per_team["n"] >= 2]

clustered = per_team.groupby("comp").agg(teams=("avg", "size"), avg=("avg", "mean"),
                                         appearances=("n", "sum"))
clustered = clustered[clustered["teams"] >= 3]
clustered["p"] = [stats.ttest_1samp(per_team.loc[per_team["comp"] == c, "avg"], 0).pvalue
                  for c in clustered.index]

print(f"\nCOUNTING EACH TEAM ONCE  ({len(clustered)} line-ups)")
print(f"   significant at 0.05 : {(clustered['p'] < 0.05).sum()}   "
      f"(chance would give {0.05 * len(clustered):.1f})")
print(f"   survives correction : {(clustered['p'] < 0.05 / len(clustered)).sum()}")

ranked = clustered.sort_values("avg", ascending=False)
ranked.index = [", ".join(c) for c in ranked.index]
print("\nbest and worst, once each team counts once:")
print(pd.concat([ranked.head(4), ranked.tail(3)]).to_string(float_format=lambda v: f"{v:.3f}"))

1,258 scored appearances from 629 maps

COUNTING EVERY APPEARANCE SEPARATELY  (25 line-ups)
   significant at 0.05 : 2   (chance would give 1.2)
   survives correction : 1

COUNTING EACH TEAM ONCE  (22 line-ups)
   significant at 0.05 : 1   (chance would give 1.1)
   survives correction : 0

best and worst, once each team counts once:
                                       teams    avg  appearances     p
breach, brimstone, cypher, neon, tejo     10  0.230           31 0.033
cypher, deadlock, omen, tejo, yoru         3  0.225            7 0.341
breach, brimstone, raze, tejo, vyse        4  0.225           10 0.160
breach, cypher, jett, omen, tejo           3  0.222           10 0.240
breach, cypher, jett, omen, sova           4 -0.094            9 0.518
fade, omen, raze, viper, vyse              7 -0.158           28 0.227
breach, jett, omen, tejo, viper            3 -0.168           10 0.380


**The one result that survived correction disappears once a team stops counting several
times over.** What's left is one line-up significant at the usual threshold, against 1.1
expected by chance: exactly what nothing looks like.

## 5. How tangled are line-ups with teams?

In [6]:
popular = beat.groupby("comp").size()
popular = popular[popular >= MIN_USES].index

dominance = []
for comp in popular:
    sub = beat[beat["comp"] == comp]
    dominance.append(sub.groupby("team").size().max() / len(sub))
dominance = np.array(dominance)

print(f"For the {len(popular)} line-ups used {MIN_USES}+ times, the single team that uses each")
print("one most often accounts for this share of all its appearances:\n")
print(f"   typical (median) : {np.median(dominance):.0%}")
print(f"   most tangled     : {dominance.max():.0%}")
print(f"   least tangled    : {dominance.min():.0%}")

For the 25 line-ups used 10+ times, the single team that uses each
one most often accounts for this share of all its appearances:

   typical (median) : 23%
   most tangled     : 90%
   least tangled    : 15%


## What this notebook found

1. **No evidence any line-up wins more than its agents deserve.** Of 54 line-ups used 10+
   times, two looked significant against 2.7 expected by chance, and none survived correction.
2. **The apparent exceptions are teams.** Paper Rex play 20 of the best-looking line-up's
   43 maps; NRG won all eight with another.
3. **It holds after adjusting for player strength** and counting each team once.
4. **But the data could only find a huge effect**: 784 maps are needed and the best has 86.
   So this isn't proof synergy doesn't exist. At this sample size, *who* plays swamps *what*
   they play.

**What would help:** more seasons (2024 now has dates, though agents and maps change between
years), pairs of agents instead of whole line-ups (far more data per combination), and
opponent-adjusted ratings.